In [29]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [30]:
### Read all the pdfs inside the directory

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files in the directory
    pdf_files = list(pdf_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in the directory.")

    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file}")
        try:           
            loader = PyPDFLoader(pdf_file)
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
                doc.metadata["Author"] = "Vishnu"

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    print(f"total documents processed so far: {len(all_documents)}")
    return all_documents
    
    
all_pdfs_documents = process_all_pdfs("../data")

Found 2 PDF files in the directory.
Processing file: ..\data\Claude_Exam_Guide.pdf
  ✓ Loaded 40 pages
Processing file: ..\data\Netflix_Data_Engineer_interview_guidelines.pdf
  ✓ Loaded 5 pages
total documents processed so far: 45


In [31]:
all_pdfs_documents[44].metadata

{'producer': 'airSlate inc. Mellivora 3.1.3.1',
 'creator': 'Microsoft® Word 2016',
 'creationdate': "D:20250115193257Z00'00'",
 'moddate': "D:20250115193257Z00'00'",
 'author': 'Shubham',
 'owner': 'shubham199827@gmail.com',
 'documentid': '218C-67D5-ACCA-0003',
 'reupload': 'f5Qnec+jplBP8MHiobpEBc1pjZf+6KZ7PHydYOeSc6sVKtVHTZw+fC8EAC14LtPpnm/xgLWaEq5aiTlfDbSE1xB/WzfwdbuvX4Z89KdSgnAXriiga2MzXGVeb/zh2QYumeo95P9pjr9+P29Hcg3pNxMG',
 'source': '..\\data\\Netflix_Data_Engineer_interview_guidelines.pdf',
 'total_pages': 5,
 'page': 4,
 'page_label': '5',
 'source_file': 'Netflix_Data_Engineer_interview_guidelines.pdf',
 'file_type': 'pdf',
 'Author': 'Vishnu'}

In [32]:
### Text splitting into chunks 

def split_documents_into_chunks(documents, chunk_size=1000, chunk_overlap=200):
    # split documents into smaller chunks for better RAG performance
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len, separators=["\n\n", "\n", " ", ""])
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")
    
    # show an example of a chunk
    if split_docs:
       print(f"\n Example chunk:")
       print(f"Content: {split_docs[0].page_content[:500]}...")  # Print the first 500 characters of the first chunk
       print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [33]:
chunks = split_documents_into_chunks(all_pdfs_documents)

Split 45 documents into 142 chunks.

 Example chunk:
Content: Claude  Certified  Architect  –  Foundations  Certification  Exam  Guide  
Introduction  The  Claude  Certiﬁed  Architect  –  Foundations  certiﬁcation  validates  that  practitioners  can  make  
informed
 
decisions
 
about
 
tradeoffs
 
when
 
implementing
 
real-world
 
solutions
 
with
 
Claude.
 
This
 
exam
 
tests
 
foundational
 
knowledge
 
across
 
Claude
 
Code,
 
the
 
Claude
 
Agent
 
SDK,
 
the
 
Claude
 
API,
 
and
 
Model
 
Context
 
Protocol
 
(MCP)
 
—
 
the
 
core
 
technolog...
Metadata: {'producer': 'Skia/PDF m148 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Claude Certified Architect – Foundations Certification Exam Guide', 'source': '..\\data\\Claude_Exam_Guide.pdf', 'total_pages': 40, 'page': 0, 'page_label': '1', 'source_file': 'Claude_Exam_Guide.pdf', 'file_type': 'pdf', 'Author': 'Vishnu'}


## Embedding and VectorstoreDB

In [34]:
import numpy as np
from sentence_transformers import SentenceTransformer       
import uuid
import chromadb
from chromadb.config import Settings
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

### Embeddings

In [35]:
## Embedding class

class EmbeddingManager():
    '''Handles document embedding generation using SentenceTransformer.'''
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        '''
        Initialize the embedding manager with a specific SentenceTransformer model.
        Args:
            model_name (str): The Hugging face name of the SentenceTransformer model to use for embedding generation.
        '''
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        '''Load the SentenceTransformer model.'''
        try:
            print(f"Loading model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        '''Generate embeddings for a list of texts.
        Args:
            texts (List[str]): A list of strings to generate embeddings for.
            returns:
                numpy array of embeddings with shape (len(texts), embedding_dim)
        '''
        if self.model is None:
            raise ValueError("Model is not loaded.")
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            return np.array([])


# Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4756.28it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### Vectorstore

In [36]:
class Vectorstore:

    def __init__(self, collection_name: str = "pdf_collection", persist_directory: str = "../data/vectorstore"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        '''Initialize the ChromaDB client and collection.'''
        try:
            # create persistent chromaDB client 
            os.makedirs(self.persist_directory, exist_ok=True)

            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(name = self.collection_name, metadata={"description": "PDF documents embeddings for RAG"}) 
            print(f"ChromaDB collection '{self.collection_name}' initialized successfully at '{self.persist_directory}'.")
            print(f"Current number of documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        '''Add documents and their corresponding embeddings to the collection.
        Args:
            documents (List[Any]): A list of document objects containing metadata and content.
            embeddings (np.ndarray): A numpy array of embeddings corresponding to the documents.
        '''
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents must match the number of embeddings.")
        
        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for chromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            
            # Generate a unique ID for each document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata = dict(doc.metadata)  # Convert metadata to a regular dictionary
            metadata["doc_index"] = i  # Add document index to metadata
            metadata["context_length"] = len(doc.page_content)  # Add context length to metadata
            metadatas.append(metadata)
          
            # document content and embedding
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding)

        # Add to chromaDB collection
        try:
            self.collection.add(
                ids=ids,
                documents=documents_text,
                metadatas=metadatas,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")



vectorstore = Vectorstore()
vectorstore    

ChromaDB collection 'pdf_collection' initialized successfully at '../data/vectorstore'.
Current number of documents in collection: 284


In [37]:
### convert text chunks into embeddings and add to vectorstore
texts =  [doc.page_content for doc in chunks] 

# Generate embeddings for the text chunks
embeddings = embedding_manager.generate_embeddings(texts)

# store it in the vector database
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 142 texts...


Batches: 100%|██████████| 5/5 [00:03<00:00,  1.61it/s]


Generated embeddings with shape: (142, 384)
Adding 142 documents to the vector store...
Successfully added 142 documents to the vector store.
Total documents in collection after addition: 426


### Retriever Pipeline from VectorStore



In [38]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: Vectorstore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [39]:
rag_retriever

In [44]:
rag_retriever.retrieve("How many domains are tested in the Claude exam?")

Retrieving documents for query: 'How many domains are tested in the Claude exam?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.59it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_f942aaa6_47',
  'content': 'instructing\n \nClaude\n \nto\n \nreport\n \nonly\n \nnew\n \nor\n \nstill-unaddressed\n \nissues\n \nto\n \navoid\n \nduplicate\n \ncomments\n \n-\n \nProviding  existing  test  ﬁles  in  context  so  test  generation  avoids  suggesting  duplicate  \nscenarios\n \nalready\n \ncovered\n \nby\n \nthe\n \ntest\n \nsuite\n \n-\n \nDocumenting  testing  standards,  valuable  test  criteria,  and  available  ﬁxtures  in  \nCLAUDE.md\n \nto\n \nimprove\n \ntest\n \ngeneration\n \nquality\n \nand\n \nreduce\n \nlow-value\n \ntest\n \noutput\n   \nAnthropic,\n \nPBC\n   \n·\n   \nConfidential\n \nNeed\n \nto\n \nKnow\n \n(NTK)',
  'metadata': {'doc_index': 47,
   'creationdate': '',
   'title': 'Claude Certified Architect – Foundations Certification Exam Guide',
   'creator': 'PyPDF',
   'source_file': 'Claude_Exam_Guide.pdf',
   'total_pages': 40,
   'producer': 'Skia/PDF m148 Google Docs Renderer',
   'source': '..\\data\\Claude_Exam_Guide.pdf',
   '

### Integration Vectordb Context pipeline with LLM output


In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = "Enter your GROQ_API_KEY here"  # Replace with your actual GROQ API key

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-120b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=10):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [57]:
answer=rag_simple("What are the content areas in the Claude exam?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What are the content areas in the Claude exam?'
Top K: 10, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 9 documents (after filtering)


The exam’s content is organized around four core pillars:

1. **Claude Code**  
2. **Claude Agent SDK**  
3. **Claude API**  
4. **Model Context Protocol (MCP)**


### Enhanced RAG Pipeline Features

In [58]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.21it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 


In [59]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 52.84it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)



Final Answer: No relevant context found.
Summary: The reply indicates that there is no applicable information or background related to the query. As a result, no further details can be provided.
History: {'question': 'what is attention is all you need', 'answer': 'No relevant context found.', 'sources': [], 'summary': 'The reply indicates that there is no applicable information or background related to the query. As a result, no further details can be provided.'}
